# Fermi-Hubbard 1D: Jordan-Wigner verification notebook

This notebook walks through the `FermiHubbard1D` implementation step by step, checking the physics at each stage. The two JW orderings — **stacked** (`[↑_0...↑_{L-1}, ↓_0...↓_{L-1}]`) and **interleaved** (`[↑_0, ↓_0, ↑_1, ↓_1, ...]`) — encode the same Hamiltonian in different qubit layouts; every test in this notebook exploits a prediction that either must hold for both, or that distinguishes them on purpose.

The verification hierarchy follows `docs/jw_fermi_hubbard_1d.md §7`:
1. Hermiticity — basic sanity
2. Encoding agreement — catches string/ordering bugs
3. Analytical dimer — catches global sign/coefficient errors
4. Free-fermion limit — tests U-term and spin decoupling
5. Electron-number symmetry — tests number conservation
6. Trotter convergence — tests the Trotterization grouping

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from qaravan.core.hamiltonians import FermiHubbard1D
from qaravan.backends.statevector import Statevector, StatevectorSimulator
from qaravan.core.observables import Magnetization
from qaravan.plot import set_style

set_style()

## 1. Construction and Hamiltonian matrix

For `num_fermions=L`, both orderings produce a `2L`-qubit system (one qubit per fermionic mode). The Hamiltonian matrix is `2^(2L) × 2^(2L)`. Here `L=3` gives 6 qubits → 64×64.

**Check:** `H = H†` to machine precision. This must hold for any `(t, U, mu)` because the Hamiltonian is Hermitian by construction; a failed check would mean a coefficient or sign error in the Pauli string assembly.

In [ ]:
fh_s = FermiHubbard1D(num_fermions=3, t=1.0, U=2.0, mu=0.0, ordering="stacked")
fh_i = FermiHubbard1D(num_fermions=3, t=1.0, U=2.0, mu=0.0, ordering="interleaved")

H_s = fh_s.as_matrix()
H_i = fh_i.as_matrix()

print(f"stacked    H shape: {H_s.shape}")
print(f"interleaved H shape: {H_i.shape}")
print()
print(f"stacked    max|H - H†|: {np.max(np.abs(H_s - H_s.conj().T)):.2e}")
print(f"interleaved max|H - H†|: {np.max(np.abs(H_i - H_i.conj().T)):.2e}")

## 2. Encoding agreement — identical eigenspectrum

Both orderings are unitary isomorphisms of the same Hilbert space: `H_interleaved = U H_stacked U†` for some permutation unitary `U`. The full spectrum is therefore identical, to machine precision, for any `(t, U)`. This is the sharpest single correctness test — it catches string length bugs, index off-by-ones, and missing h.c. terms.

**What it misses:** a sign error that is *common* to both encoders (e.g. flipping the overall hopping sign `t → -t`). That would require an independent anchor (the dimer check in §3 provides one).

In [ ]:
params = [(1.0, 0.0), (1.0, 2.0), (0.5, 4.0), (2.0, 1.0)]

print(f"{'t':>6}  {'U':>6}  {'E0 stacked':>14}  {'E0 interleaved':>16}  {'max |diff|':>12}")
print("-" * 62)
for t, U in params:
    fh_s = FermiHubbard1D(num_fermions=3, t=t, U=U, mu=0.0, ordering="stacked")
    fh_i = FermiHubbard1D(num_fermions=3, t=t, U=U, mu=0.0, ordering="interleaved")
    E_s = np.sort(np.linalg.eigvalsh(fh_s.as_matrix()))[:5]
    E_i = np.sort(np.linalg.eigvalsh(fh_i.as_matrix()))[:5]
    diff = np.max(np.abs(E_s - E_i))
    print(f"{t:>6.2f}  {U:>6.2f}  {E_s[0]:>14.8f}  {E_i[0]:>16.8f}  {diff:>12.2e}")

print()
print("Lowest 5 eigenvalues for (t=1, U=2):")
fh_s = FermiHubbard1D(num_fermions=3, t=1.0, U=2.0, mu=0.0, ordering="stacked")
fh_i = FermiHubbard1D(num_fermions=3, t=1.0, U=2.0, mu=0.0, ordering="interleaved")
E_s5 = np.sort(np.linalg.eigvalsh(fh_s.as_matrix()))[:5]
E_i5 = np.sort(np.linalg.eigvalsh(fh_i.as_matrix()))[:5]
print(f"{'Level':>6}  {'stacked':>14}  {'interleaved':>14}")
for k, (es, ei) in enumerate(zip(E_s5, E_i5)):
    print(f"{k:>6}  {es:>14.8f}  {ei:>14.8f}")

## 3. Hubbard dimer: analytical ground-state energy

For `L=2` sites at half-filling (`n_electrons=2`), the Hubbard dimer ground-state energy has an exact closed form derivable from the 4×4 Hamiltonian restricted to the two-particle sector:

$$E_\mathrm{gs} = \frac{U - \sqrt{U^2 + 16t^2}}{2}$$

Limiting cases:
- `U=0`: `E_gs = -2t sqrt(2) ≈ -2.83t` (two bonding electrons in a tight-binding dimer, each contributing `-sqrt(2)t`)  
  Wait — actually at half-filling with 2 electrons sharing one spin each: `E_gs = (0 - sqrt(0 + 16t^2))/2 = -2t`. Let's verify.
- `U >> t`: `E_gs ≈ -4t²/U` (superexchange; dominant singlet energy)

Both orderings must agree with this formula. `fh.ground_state(n_electrons=2)` restricts the diagonalization to the fixed-`N=2` sector.

In [ ]:
test_cases = [(1.0, 2.0), (0.5, 4.0), (1.0, 0.0)]

print(f"{'t':>5}  {'U':>5}  {'E_exact':>12}  {'stacked':>12}  {'interleaved':>12}  {'err_s':>10}  {'err_i':>10}")
print("-" * 80)
for t, U in test_cases:
    E_exact = (U - np.sqrt(U**2 + 16 * t**2)) / 2
    for ordering in ["stacked", "interleaved"]:
        fh = FermiHubbard1D(num_fermions=2, t=t, U=U, mu=0.0, ordering=ordering)
        sv = fh.ground_state(n_electrons=2)
        energy = sv.expectation(fh.as_observable()).real
        err = abs(energy - E_exact)
        if ordering == "stacked":
            E_s, err_s = energy, err
        else:
            print(f"{t:>5.2f}  {U:>5.2f}  {E_exact:>12.8f}  {E_s:>12.8f}  {energy:>12.8f}  {err_s:>10.2e}  {err:>10.2e}")

## 4. Free-fermion limit (`U=0`)

At `U=0` the model decouples into two independent tight-binding chains, one per spin. For an open chain of `L` sites the single-particle levels are:

$$\varepsilon_k = -2t \cos\!\left(\frac{\pi k}{L+1}\right), \quad k = 1, \ldots, L$$

**Stacked spin-decoupling check.** In the stacked register, qubits `0..L-1` carry `↑` and qubits `L..2L-1` carry `↓`. At `U=0` these sectors are completely decoupled, so the global ground state factorises: `|GS⟩ = |GS_↑⟩ ⊗ |GS_↓⟩`. Consequently the reduced density matrix `ρ_↓ = Tr_↑[|GS⟩⟨GS|]` must be a *pure* state: `Tr[ρ_↓²] = 1`. This is a falsifiable prediction that directly tests whether the `U=0` stacked Hamiltonian has the correct (zero) coupling between spin sectors.

For interleaved the analogous cut separates spin sectors non-trivially, so we do *not* expect purity there.

In [ ]:
num_fermions = 4
t = 1.0

# Open tight-binding spectrum for one spin sector
tb_levels = sorted(-2 * t * np.cos(np.pi * k / (num_fermions + 1)) for k in range(1, num_fermions + 1))
print("Tight-binding single-particle levels (one spin):")
print([f"{e:.5f}" for e in tb_levels])

# Ground state of each ordering at U=0
fh_s = FermiHubbard1D(num_fermions=num_fermions, t=t, U=0.0, mu=0.0, ordering="stacked")
fh_i = FermiHubbard1D(num_fermions=num_fermions, t=t, U=0.0, mu=0.0, ordering="interleaved")

# Plot tight-binding spectrum vs FH spectrum for stacked ordering
H_s = fh_s.as_matrix()
evals_s = np.sort(np.linalg.eigvalsh(H_s))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(len(tb_levels)), tb_levels, "o", ms=8, label="tight-binding (1 spin)")
axes[0].set_xlabel("level index")
axes[0].set_ylabel("energy / t")
axes[0].set_title(f"Single-particle levels (L={num_fermions}, U=0)")
axes[0].legend()

# purity of down-sector RDM for stacked vs interleaved
sv_s = fh_s.ground_state()
sv_i = fh_i.ground_state()

down_sites_stacked = list(range(num_fermions, 2 * num_fermions))
rho_down_s = sv_s.rdm(down_sites_stacked)
purity_s = np.trace(rho_down_s @ rho_down_s).real

# For interleaved, down-qubits are at odd indices
down_sites_interleaved = list(range(1, 2 * num_fermions, 2))
rho_down_i = sv_i.rdm(down_sites_interleaved)
purity_i = np.trace(rho_down_i @ rho_down_i).real

purities = [purity_s, purity_i]
labels = ["stacked\n(pure: Tr[rho^2]=1)", "interleaved\n(mixed: Tr[rho^2]<1)"]
axes[1].bar(labels, purities, color=["steelblue", "coral"])
axes[1].axhline(1.0, ls="--", color="gray", lw=1)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Tr[rho_down^2]")
axes[1].set_title("Down-sector RDM purity at U=0")

plt.tight_layout()
plt.show()

print(f"Stacked  Tr[rho_down^2] = {purity_s:.10f}  (expected 1.0)")
print(f"Interleaved Tr[rho_down^2] = {purity_i:.10f}  (expected < 1)")

## 5. Electron number conservation

`[H, N] = 0` where `N = sum_k (I - Z_k)/2` is the total electron number operator. Both encodings must conserve electron number: the ground state in the fixed-`N=n_electrons` sector must satisfy `<N> = n_electrons` exactly.

This catches particle-number-violating terms — e.g. if a hopping term was missing its h.c., it would create or destroy electrons.

The number operator in the computational basis is diagonal: element `i` equals the Hamming weight of `i` (number of `|1⟩` bits).

In [ ]:
num_fermions = 3
num_qubits = 2 * num_fermions

# Number operator: diagonal with popcount(i) as eigenvalues
n_op_diag = np.array([float(bin(i).count("1")) for i in range(2**num_qubits)])

print(f"{'ordering':>12}  {'n_elec':>8}  {'<N>':>14}  {'|<N> - n_elec|':>16}")
print("-" * 55)
for ordering in ["stacked", "interleaved"]:
    fh = FermiHubbard1D(num_fermions=num_fermions, t=1.0, U=2.0, mu=0.0, ordering=ordering)
    for n_elec in [2, 3, 4]:
        sv = fh.ground_state(n_electrons=n_elec)
        arr = sv.to_array()
        expected_N = np.real(np.dot(arr.conj(), n_op_diag * arr))
        err = abs(expected_N - n_elec)
        print(f"{ordering:>12}  {n_elec:>8}  {expected_N:>14.10f}  {err:>16.2e}")

## 6. Trotter evolution

### 6a. Dynamics: Neel-like initial state

Start from `|101010⟩` (bitstring in computational basis for 6 qubits: alternating occupied/empty, a Neel-like product state in the stacked ordering where qubits 0,1,2 are up-spins and 3,4,5 are down-spins). Evolve under stacked FH with `order=2`, `dt=0.05`, 40 steps. Track `<M_z> = (1/n) sum_i Z_i` of the full 6-qubit system.

The magnetization measures net `(n_unoccupied - n_occupied)` — it starts positive (3 occupied out of 6) and should oscillate as the state evolves under the hopping and interaction terms.

### 6b. Trotter error scaling

At `dt → 0`, the one-step Frobenius error between `U_Trotter(dt)` and `e^{-i H dt}` should scale as:
- `O(dt^2)` for order-1 (first-order Suzuki-Trotter)
- `O(dt^3)` for order-2 (symmetric/second-order)

A wrong grouping (non-commuting terms placed in the same Trotter layer) breaks the error bound and would show a slope < 2. The log-log plot should show straight lines with slopes ≈2 and ≈3.

In [ ]:
num_fermions = 3
fh = FermiHubbard1D(num_fermions=num_fermions, t=1.0, U=2.0, mu=0.0, ordering="stacked")

# --- 6a: real-time dynamics ---
dt, num_steps = 0.05, 40
step_circ = fh.trotter_circuit(dt, order=2)

# Neel-like: |101010> in 6 qubits
sv = Statevector(bitstring="101010")
mag_obs = Magnetization(2 * num_fermions, axis="Z")

times = [0.0]
mags = [sv.expectation(mag_obs).real]
for step in range(num_steps):
    sv = sv.apply(step_circ)
    times.append((step + 1) * dt)
    mags.append(sv.expectation(mag_obs).real)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(times, mags)
axes[0].set_xlabel("t")
axes[0].set_ylabel(r"$\langle M_z \rangle$")
axes[0].set_title("FH dynamics from |101010> (stacked, order-2)")

# --- 6b: Trotter error scaling ---
dt_values = np.array([0.1, 0.07, 0.05, 0.03, 0.02, 0.01])
H_mat = fh.as_matrix()

errors_order1, errors_order2 = [], []
for dt in dt_values:
    exact_prop = expm(-1j * dt * H_mat)
    trotter1 = fh.trotter_circuit(dt, order=1).to_matrix()
    trotter2 = fh.trotter_circuit(dt, order=2).to_matrix()
    errors_order1.append(np.linalg.norm(trotter1 - exact_prop, ord="fro"))
    errors_order2.append(np.linalg.norm(trotter2 - exact_prop, ord="fro"))

slope1 = np.polyfit(np.log(dt_values), np.log(errors_order1), 1)[0]
slope2 = np.polyfit(np.log(dt_values), np.log(errors_order2), 1)[0]

axes[1].loglog(dt_values, errors_order1, "o-", label=f"order-1 (slope={slope1:.2f}, expected 2)")
axes[1].loglog(dt_values, errors_order2, "s-", label=f"order-2 (slope={slope2:.2f}, expected 3)")
axes[1].set_xlabel("dt")
axes[1].set_ylabel("Frobenius error")
axes[1].set_title("Trotter error scaling (L=3, stacked)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"order-1 log-log slope: {slope1:.3f}  (expected ~2.0)")
print(f"order-2 log-log slope: {slope2:.3f}  (expected ~3.0)")